<img src="http://hilpisch.com/tpq_logo.png" alt="The Python Quants" width="35%" align="right" border="0"><br>

# Certificate Programs

**Mentoring Session**

## Imports

In [ ]:
!git clone https://github.com/tpq-classes/python_for_algo_trading_addon.git
import sys
sys.path.append('python_for_algo_trading_addon')


In [ ]:
import numpy as np
import pandas as pd
from pylab import plt
plt.style.use('seaborn-v0_8')

## Streaming Data

In [ ]:
import tpqoa

In [ ]:
api = tpqoa.tpqoa('../oanda.cfg')

In [ ]:
api.stream_data?

In [ ]:
api.stream_data('BTC_USD', stop=10)

In [ ]:
# api.stream_data??

In [ ]:
api.on_success??

In [ ]:
# api.__init__??

In [ ]:
class mytpqoa(tpqoa.tpqoa):
    def __init__(self, conf_file):
        super().__init__(conf_file)
        self.raw = pd.DataFrame()
    def on_success(self, time, bid, ask):
        print(time, bid, ask)
        df = pd.DataFrame({'bid': bid, 'ask': ask},
                          index=[pd.Timestamp(time)])
        self.raw = self.raw.append(df)
        self.data = self.raw.resample('3s', label='right').last().ffill()

In [ ]:
api = mytpqoa('../oanda.cfg')

In [ ]:
api.stream_data('BTC_USD', stop=15)

In [ ]:
api.raw.info()

In [ ]:
api.data

In [ ]:
api.raw.plot();

In [ ]:
api.data['bid'].plot();

## Market Prediction

### The Data

In [ ]:
url = 'http://hilpisch.com/pyalgo_eikon_eod_data.csv'

In [ ]:
raw = pd.read_csv(url, index_col=0, parse_dates=True).dropna()

In [ ]:
symbols = ['AAPL.O', 'MSFT.O']

In [ ]:
data = raw[symbols]

In [ ]:
data.head()

### The Features

In [ ]:
rets = np.log(data / data.shift(1))
rets.head()

In [ ]:
lags = 3
cols = list()
for sym in rets.columns:
    for lag in range(1, lags + 1):
        col = f'{sym}_lag_{lag}'
        rets[col] = rets[sym].shift(lag)
        cols.append(col)

In [ ]:
rets.head()

In [ ]:
rets.dropna(inplace=True)

### Data Split

In [ ]:
split = int(len(rets) * 0.8)
split

In [ ]:
train = rets.iloc[:split].copy()

In [ ]:
mu, std = train.mean(), train.std()

In [ ]:
train_ = (train - mu) / std

In [ ]:
test = rets.iloc[split:].copy()

In [ ]:
test_ = (test - mu) / std

### The Training

In [ ]:
from sklearn.neural_network import MLPRegressor

In [ ]:
model = MLPRegressor(hidden_layer_sizes=[24, 24], shuffle=False,
                    validation_fraction=0.1, random_state=100)

In [ ]:
model.fit(train_[cols], train[symbols])

### The Prediction

In [ ]:
h = [sym + '_pred' for sym in symbols]

In [ ]:
test[h] = model.predict(test_[cols])

**Exercise**: Transform the whole approach to a classification problem and calculate the accuracy scores. What impact on the accuracy does the normalization have?

## Natural Language Processing

`pip install PyPDF2`

In [ ]:
import nlp
import urllib
import requests
import PyPDF2 as pdf

In [ ]:
url = 'http://tpq.io'

In [ ]:
url = 'https://certificate.tpq.io/cert_study_plan.pdf'
url = 'https://arxiv.org/pdf/2102.09370.pdf'

In [ ]:
with urllib.request.urlopen(url) as response:
    with open('file.pdf', 'wb') as f:
        f.write(response.read())

In [ ]:
f = pdf.PdfFileReader('file.pdf')

In [ ]:
f

In [ ]:
f.getNumPages()

In [ ]:
t = list()
for page in range(f.getNumPages()):
    t.append(f.getPage(page).extractText())

In [ ]:
t = nlp.clean_up_text(' '.join(t))

In [ ]:
nlp.generate_key_words(t, 10)

In [ ]:
nlp.generate_word_cloud(t, 30)

<img src="http://hilpisch.com/tpq_logo.png" alt="The Python Quants" width="30%" align="right" border="0"><br>

<a href="http://tpq.io" target="_blank">http://tpq.io</a> | <a href="http://twitter.com/dyjh" target="_blank">@dyjh</a> | <a href="mailto:training@tpq.io">training@tpq.io</a>